# 01 – Exploration des données : gravité des accidents corporels (BAAC)
**Objectif** : classification de la gravité (`grav`) d'un usager impliqué dans un accident corporel.
**Source** : data.gouv.fr – BAAC 2005-2024 (Licence Ouverte). Année de travail : 2023 (à confirmer en équipe).
> Consigne : 10-20 features, quelques 10-100K instances, numériques + catégorielles, pas de séries temporelles.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
YEAR = 2023
RAW = Path("../data/raw")/str(YEAR)

## 1. Chargement et fusion des 4 tables

In [ ]:
def load(kind):
    return pd.read_csv(RAW/f"{kind}-{YEAR}.csv", sep=";", dtype=str, low_memory=False)
car, lieux, usagers, veh = (load(k) for k in ["caracteristiques","lieux","usagers","vehicules"])
for n,d in [("caracteristiques",car),("lieux",lieux),("usagers",usagers),("vehicules",veh)]:
    print(n, d.shape)

In [ ]:
df = (usagers.merge(veh, on=["Num_Acc","id_vehicule","num_veh"], how="left", suffixes=("","_veh"))
             .merge(car, on="Num_Acc", how="left")
             .merge(lieux, on="Num_Acc", how="left"))
print(df.shape); df.head()

## 2. Métadonnées, types et valeurs manquantes\n(Les codes `-1` et vides signifient « non renseigné » dans les BAAC.)

In [ ]:
df = df.replace({"-1": np.nan, " -1": np.nan, "": np.nan})
info = pd.DataFrame({"nunique": df.nunique(), "na_%": (df.isna().mean()*100).round(1)}).sort_values("na_%", ascending=False)
info

## 3. Variable cible `grav`\n1 = indemne, 2 = tué, 3 = blessé hospitalisé, 4 = blessé léger

In [ ]:
labels = {"1":"Indemne","2":"Tué","3":"Hospitalisé","4":"Léger"}
y = df["grav"].map(labels)
print(y.value_counts(normalize=True).round(3))
y.value_counts().plot.bar(title="Distribution de la gravité"); plt.show()

## 4. Distributions des variables clés

In [ ]:
df["an_nais"] = pd.to_numeric(df["an_nais"], errors="coerce")
df["age"] = YEAR - df["an_nais"]
df["heure"] = pd.to_numeric(df["hrmn"].str[:2], errors="coerce")
df["vma"] = pd.to_numeric(df["vma"], errors="coerce")
fig, ax = plt.subplots(1,3, figsize=(16,4))
sns.histplot(df["age"].dropna(), bins=50, ax=ax[0]); ax[0].set_title("Âge")
sns.histplot(df["heure"].dropna(), bins=24, ax=ax[1]); ax[1].set_title("Heure")
sns.histplot(df["vma"].dropna(), bins=20, ax=ax[2]); ax[2].set_title("Vitesse maximale autorisée")
plt.tight_layout(); plt.show()

In [ ]:
cats = ["lum","atm","catr","surf","catu","sexe","catv","agg","col"]
fig, axes = plt.subplots(3,3, figsize=(16,11))
for a,c in zip(axes.ravel(), cats):
    pd.crosstab(df[c], y, normalize="index").plot.bar(stacked=True, ax=a, legend=False, title=f"grav | {c}")
axes[0,0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 5. Corrélations, clustering non supervisé, conclusions\nÀ compléter : matrice de corrélation (Cramér's V pour les catégorielles), PCA/UMAP, K-Means vs `grav`, décisions de preprocessing.